# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the FAIR^2 mass spectrometry dataset using the `mlcroissant` library.

### Dataset Source
Croissant schema URL: https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json

In [ ]:
# Install mlcroissant if not already available
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and prepare for exploration.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
# The metadata object has dataset-level information
print('Dataset Title:', dataset.metadata.name)
print('Description:', dataset.metadata.description)
print('Version:', dataset.metadata.version)
print('Date Published:', getattr(dataset.metadata, 'datePublished', None))

## 2. Data Overview

List available record sets, and for each, list their fields and columns, always referencing `@id` values where applicable.

In [ ]:
from collections import OrderedDict
def display_record_sets(dataset):
    print("Available record sets (by @id):\n------------------------------")
    for rs in dataset.metadata.record_sets:
        print(f"- @id: {rs['@id']}")
        print(f"  Name: {rs.get('name','')}")
        print("  Fields:")
        for field in rs.get('fields', []):
            print(f"    - @id: {field['@id']}, name: {field.get('name','')}, dataType: {field.get('dataType','')}")
        print("  Columns:")
        for c in rs.get('columns', []):
            print(f"    - @id: {c['@id']}, name: {c.get('name','')}")
        print()

display_record_sets(dataset)

In [ ]:
# For demonstration, list the first 2 records in EACH record set
for rs in dataset.metadata.record_sets:
    print(f"Sample records for record_set @id: {rs['@id']}")
    for i, rec in enumerate(dataset.records(record_set=rs['@id'])):
        pprint.pprint(rec)
        if i >= 1:
            break
    print('-' * 40)

## 3. Data Extraction

Load the tables from the main data record sets into pandas DataFrames, accessing by record set `@id`. This makes data manipulation and analysis simple.

In [ ]:
# Extract all record sets to DataFrames and display their columns
record_set_ids = [rs['@id'] for rs in dataset.metadata.record_sets]
dataframes = {}

for rsid in record_set_ids:
    records = list(dataset.records(record_set=rsid))
    df = pd.DataFrame(records)
    dataframes[rsid] = df
    print(f"Record set @id: {rsid}")
    print("Fields:", df.columns.tolist())
    print(f"Number of records: {len(df)}\n")

# For illustration, pick the first record set
example_record_set_id = record_set_ids[0]
print(f"Example DataFrame for record set {example_record_set_id}:")
dataframes[example_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, using field `@id` references.

We'll select a numeric field (e.g., 'MW [kDa]' for molecular weight, or 'Coverage [%]' for protein coverage, or peptide counts if present) and a group field (e.g., description or some categorical sample/group field) using their `@id`.

In [ ]:
# For the example, let's select the first record set
record_set_id = example_record_set_id
df = dataframes[record_set_id]

# Try to infer a good numeric field for the example, fall back if not standard
possible_numeric_fields = [
    'MW [kDa]', 'Coverage [%]', 'Peptides', 'Unique peptides', 'pI', 'Abundance',
    # Also match with typical Croissant @id style fields (i.e., use dataset.metadata fields)
]
# Find numeric column in DataFrame
numeric_field = None
for col in df.columns:
    if any(x.lower() in col.lower() for x in possible_numeric_fields):
        numeric_field = col
        break
# If not found, just pick the first numeric column
if numeric_field is None:
    for col in df.select_dtypes(include=['number']).columns:
        numeric_field = col
        break

print(f"Selected numeric field: {numeric_field}")

# Pick a group field - e.g., 'Description' or 'Sample' (or similar). If not found, will skip grouping
possible_group_fields = ['description', 'Description', 'Group', 'Sample', 'Condition']
group_field = None
for col in df.columns:
    if any(x.lower() == col.lower() for x in possible_group_fields):
        group_field = col
        break

# Filter: Select rows with numeric_field > threshold
if numeric_field:
    # Convert column to numeric, errors='coerce' will make non-convertibles NaN
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
    threshold = df[numeric_field].mean() if df[numeric_field].mean() > 0 else 10
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records where {numeric_field} > {threshold:.2f} (using mean as threshold, or fallback 10)")
    print(filtered_df[[numeric_field]].head())

    # Normalize
    filtered_df[numeric_field + '_normalized'] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(filtered_df[[numeric_field, numeric_field + '_normalized']].head())
    
    # Optionally, group by a field
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped mean of {numeric_field} by {group_field}:")
        print(grouped_df.head())
else:
    print("No suitable numeric field found for EDA.")

## 5. Visualization

Plot a histogram of the selected numeric field and, if available, its normalized values. If a group field is found, plot the mean value per group.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), bins=20)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()

    if f"{numeric_field}_normalized" in filtered_df.columns:
        plt.figure(figsize=(8,4))
        sns.histplot(filtered_df[f"{numeric_field}_normalized"].dropna(), bins=20, color='orange')
        plt.title(f"Distribution of Normalized {numeric_field} (filtered)")
        plt.xlabel(f"{numeric_field} (normalized)")
        plt.ylabel('Frequency')
        plt.show()

    if group_field and 'grouped_df' in locals():
        plt.figure(figsize=(10,5))
        sns.barplot(x=group_field, y=numeric_field, data=grouped_df)
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No suitable numeric field for visualization.")

## 6. Conclusion

In summary, we explored the FAIR^2 dataset describing proteins captured from stimulated human mast cell extracellular vesicles. Using `mlcroissant`, we loaded the dataset directly from its Croissant schema, inspected metadata, listed fields and columns by `@id`, extracted main data tables, and performed initial exploratory data analysis. Further analysis (such as machine learning, advanced visualizations, or joining data across record sets) can easily be performed with this pipeline.

**Tips:**
- All fields, columns, and record sets can be referenced by their `@id` attribute for robust and reproducible analysis.
- For deeper analyses, consult the dataset metadata for definitions and units.
- Revisit this workflow as the dataset is updated or extended in FAIR repositories.